In [ ]:
# !wget https://data.ncl.ac.uk/ndownloader/articles/24050256/versions/3 -O ../data/GeoClassing.zip
# !mkdir ../data/GeoClassing
# !unzip ../data/GeoClassing.zip -d ../data/GeoClassing/
# !rm ../data/GeoClassing.zip

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 60 * 60, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [ ]:
hyperparams = {
    "num_epochs": 128,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.4},
    "pruning_thresholds": {"fc0": 0.1},
    "plateau_threshold": 0.5,
    "min_epochs_between_expansions": 24,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 1000,
    "weight_decay": 0,
    "diff_threshold": 0.05,
}

In [ ]:
train_x = np.load("../data/GeoClassing/train_x.npy")
train_y = np.load("../data/GeoClassing/train_y.npy")
valid_x = np.load("../data/GeoClassing/valid_x.npy")
valid_y = np.load("../data/GeoClassing/valid_y.npy")
test_x = np.load('../data/GeoClassing/test_x.npy')
test_y = np.load('../data/GeoClassing/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 3 * 60 * 60)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 3 * 60 * 60)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 3 * 60 * 60)
test_tensor_y = torch.tensor(test_y).long()


train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = max(1, int(finder_fraction * len(train_dataset)))
finder_dataset = Subset(train_dataset, range(n_finder))

bs = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
finder_loader = DataLoader(finder_dataset, batch_size=bs, shuffle=False)
val_loader = DataLoader(valid_dataset, batch_size=bs, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False)

In [ ]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

In [ ]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="geoclassing",
)
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

In [ ]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(m):
    return optim.Adam(m.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])

trained_model = train_ssonn(
    model, train_loader, val_loader, finder_loader,
    criterion, optimizer_factory, hyperparams, device, experiment, True,
)
sparse_model = trained_model

In [ ]:
experiment.end()

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [ ]:
accuracy, params

In [ ]:
torch.save(sparse_model, 'geoclassing.pt')

In [ ]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 80.33, 92.49, 93.67, 72.76, 86.00, 94.21, 89.99, 94.61, 96.03, 95.66, 95.54, 63.56, 10],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))
styled_table